# Residual CNN Traffic-Sign Study — 80% Accuracy Goal + Measured FGSM Results

**Class labels:** Stop, 35 mph, 45 mph  · **Input size:** 64 × 64 RGB · **Model:** the TensorFlow/Keras Residual CNN adapted from the UAV project.

**Two DIFFERENT 80% goals:**
- **Clean model accuracy ≥ 80%** — fraction of *previously unseen* signs correctly classified without an attack.
- **FGSM attack success rate ≥ 80%** — fraction of examples the clean model classified correctly that become wrong after perturbation.

The notebook *tries* to reach ≥80% clean **validation** accuracy, tests offline FGSM strengths on the **validation** split only, and then measures final clean accuracy and attack success on the **held-out test** split. If either goal is not met, it will say so: **results are measured, never pre-filled or guaranteed.**

All attacks occur on saved/test images; the live webcam and drone are not modified.

In [ ]:
!pip -q install kagglehub imbalanced-learn

In [ ]:
import os
import re
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Download the traffic-sign classification dataset

The notebook uses the LISAC classification dataset derived from the LISA Traffic Sign Dataset.

If KaggleHub cannot download automatically in your Colab session, upload/extract the dataset manually and set `DATASET_ROOT` to that folder.

In [ ]:
DATASET_HANDLE = "mmontiel/lisac-traffic-sign-classification-dataset"

try:
    DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print("Dataset downloaded to:", DATASET_ROOT)
except Exception as e:
    print("Automatic Kaggle download failed.")
    print("Error:", e)
    print()
    print("Download the LISAC traffic-sign classification dataset manually,")
    print("upload/extract it in Colab, then set DATASET_ROOT to that folder.")
    raise

## 2. Find Stop / 35 mph / 45 mph images

This code is intentionally tolerant of different folder layouts. It looks through every image path and identifies the class from any directory component in its path.

In [ ]:
IMG_SIZE = 64

CLASS_NAMES = ["Stop", "speedLimit35", "speedLimit45"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

ALIASES = {
    "stop": "Stop",
    "stopsign": "Stop",
    "speedlimit35": "speedLimit35",
    "35speedlimit": "speedLimit35",
    "speed35": "speedLimit35",
    "speedlimit45": "speedLimit45",
    "45speedlimit": "speedLimit45",
    "speed45": "speedLimit45",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def infer_class_from_path(path):
    for part in reversed(path.parts):
        key = normalize_name(part)
        if key in ALIASES:
            return ALIASES[key]
    return None

all_image_paths = [
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
]

selected = []
for p in all_image_paths:
    label = infer_class_from_path(p)
    if label in CLASS_TO_IDX:
        selected.append((p, label))

print("Total images found in dataset:", len(all_image_paths))
print("Selected target images:", len(selected))

for class_name in CLASS_NAMES:
    count = sum(label == class_name for _, label in selected)
    print(f"{class_name}: {count}")

if not selected:
    raise RuntimeError(
        "No matching class folders were found. "
        "Inspect DATASET_ROOT and update ALIASES if your downloaded copy uses different folder names."
    )

## 3. Load and preprocess the images

The GitHub Residual CNN uses 64×64 RGB input normalized to `[0, 1]`.

In [ ]:
X = []
y = []

for path, class_name in selected:
    img = cv2.imread(str(path))
    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0

    X.append(img)
    y.append(CLASS_TO_IDX[class_name])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)

for i, name in enumerate(CLASS_NAMES):
    print(name, int(np.sum(y == i)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for class_idx, ax in enumerate(axes):
    matches = np.where(y == class_idx)[0]
    if len(matches):
        idx = matches[0]
        ax.imshow(X[idx])
        ax.set_title(CLASS_NAMES[class_idx])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Train / validation / test split

In [ ]:
x_train, x_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

## 5. Use real training images and class weights (rather than pixel-space SMOTE)

Class weights compensate for unbalanced signs without synthesizing unrealistic pixel-space mixtures. Small rotations/translations/zoom are used for training only. **Do not flip images horizontally**, since speed-limit numerals would change.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=y_train
)
SAMPLE_WEIGHTS = np.asarray(weights, dtype=np.float32)[y_train]
print("Training class weights:", dict(zip(CLASS_NAMES, weights.round(3))))

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32

# The augmentation acts only during training. It is not applied on validation or test.
augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03, fill_mode="reflect"),
    tf.keras.layers.RandomTranslation(0.04, 0.04, fill_mode="reflect"),
    tf.keras.layers.RandomZoom(0.08, fill_mode="reflect"),
    tf.keras.layers.RandomContrast(0.12),
], name="training_augmentation")

def make_training_dataset(seed):
    ds = tf.data.Dataset.from_tensor_slices(
        (x_train, y_train, SAMPLE_WEIGHTS)
    )
    ds = ds.shuffle(len(x_train), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.map(
        lambda img, label, weight: (
            tf.clip_by_value(augment(img, training=True), 0.0, 1.0),
            label,
            weight
        ),
        num_parallel_calls=AUTOTUNE
    )
    return ds.prefetch(AUTOTUNE)

## 6. Residual CNN — same architecture as the GitHub notebook

In [ ]:
def build_residual_cnn():
    inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = tf.keras.layers.Conv2D(
        32, 3, padding="same", activation="relu"
    )(inputs)
    x = tf.keras.layers.BatchNormalization()(x)

    # Residual block
    shortcut = x

    x = tf.keras.layers.Conv2D(
        32, 3, padding="same", activation="relu"
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv2D(
        32, 3, padding="same"
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(
        64, 3, padding="same", activation="relu"
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(
        len(CLASS_NAMES),
        activation="softmax"
    )(x)

    model = tf.keras.models.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model_resnet = build_residual_cnn()
model_resnet.summary()

## 7. Train the Residual CNN

In [ ]:
# Only validation accuracy determines model selection; the test set stays untouched.
TARGET_CLEAN_ACCURACY = 0.80
SEEDS = [42, 123, 2026]
BEST_MODEL_FILE = "residual_cnn_traffic_signs.keras"

best_val_accuracy = -1.0
best_seed = None
best_history = None

for seed in SEEDS:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model_resnet = build_residual_cnn()
    model_resnet.optimizer.learning_rate.assign(0.001)

    history = model_resnet.fit(
        make_training_dataset(seed),
        validation_data=(x_val, y_val),
        epochs=35,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", mode="max", patience=8,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=3,
                min_lr=1e-5
            )
        ],
        verbose=1
    )

    _, val_accuracy = model_resnet.evaluate(x_val, y_val, verbose=0)
    print(f"Seed {seed}: validation accuracy = {val_accuracy:.2%}")

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = float(val_accuracy)
        best_seed = seed
        best_history = history.history
        model_resnet.save(BEST_MODEL_FILE)

    if best_val_accuracy >= TARGET_CLEAN_ACCURACY:
        print("Clean VALIDATION accuracy goal reached.")
        break

model_resnet = tf.keras.models.load_model(BEST_MODEL_FILE)
print(f"Best model seed: {best_seed}")
print(f"Best validation accuracy: {best_val_accuracy:.2%}")
if best_val_accuracy < TARGET_CLEAN_ACCURACY:
    print("GOAL NOT MET. Improve data quality/quantity or training before claiming 80% accuracy.")

## 8. Report clean classification performance on untouched test images

**This is the actual clean accuracy**. A validation score ≥80% does not guarantee ≥80% on the test set.

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import pandas as pd

clean_probs = model_resnet.predict(x_test, batch_size=64, verbose=0)
clean_pred = np.argmax(clean_probs, axis=1)
clean_test_acc = accuracy_score(y_test, clean_pred)
correct_clean_mask = clean_pred == y_test
correct_clean_count = int(correct_clean_mask.sum())

print("Clean TEST accuracy:", f"{clean_test_acc:.2%}")
print("Number of clean-correct test images:", correct_clean_count)
print(classification_report(
    y_test, clean_pred, target_names=CLASS_NAMES, zero_division=0
))

if correct_clean_count == 0:
    raise RuntimeError("No test samples were classified correctly; FGSM success rate is undefined.")

## 9. Offline FGSM on the validation split (choose epsilon without seeing test results)

We measure both **attacked accuracy on all images** and **attack success rate on initially correct images**. The latter is the correct metric for an untargeted misclassification attack.

Epsilon controls the maximum pixel change on input values normalized to [0,1]. Larger epsilon may create *visible* image changes; it is not automatically a realistic physical-world attack.

In [ ]:
def fgsm_attack_batch(model, images, labels, epsilon):
    x = tf.convert_to_tensor(images, dtype=tf.float32)
    y = tf.convert_to_tensor(labels, dtype=tf.int64)
    with tf.GradientTape() as tape:
        tape.watch(x)
        predictions = model(x, training=False)
        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(y, predictions)
        )
    grad = tape.gradient(loss, x)
    if grad is None:
        raise RuntimeError("Input gradient was None")
    return tf.clip_by_value(x + epsilon * tf.sign(grad), 0.0, 1.0)


def predict_with_fgsm(model, images, labels, epsilon, batch_size=64):
    outputs = []
    for start in range(0, len(images), batch_size):
        end = min(start + batch_size, len(images))
        batch_x = images[start:end]
        batch_y = labels[start:end]
        if epsilon == 0:
            adv_x = batch_x
        else:
            adv_x = fgsm_attack_batch(
                model, batch_x, batch_y, epsilon
            ).numpy()
        outputs.append(model.predict(adv_x, verbose=0))
    return np.concatenate(outputs, axis=0)

# No live images / camera feeds are attacked.
EPSILONS = [0., 1/255, 2/255, 4/255, 8/255, 12/255, 16/255]
TARGET_ATTACK_SUCCESS = 0.80

val_clean_prob = model_resnet.predict(x_val, verbose=0)
val_clean_pred = np.argmax(val_clean_prob, axis=1)
val_correct_mask = val_clean_pred == y_val
val_clean_acc = accuracy_score(y_val, val_clean_pred)
if val_correct_mask.sum() == 0:
    raise RuntimeError("Model has zero validation correct predictions.")

validation_rows = []
for eps in EPSILONS:
    adv_probs = predict_with_fgsm(model_resnet, x_val, y_val, eps)
    adv_pred = adv_probs.argmax(axis=1)
    row = {
        "epsilon": float(eps),
        "validation_attacked_accuracy": float(accuracy_score(y_val, adv_pred)),
        "validation_attack_success_rate": float(np.mean(
            adv_pred[val_correct_mask] != y_val[val_correct_mask]
        )),
    }
    validation_rows.append(row)

validation_df = pd.DataFrame(validation_rows)
print(validation_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Select the smallest epsilon meeting 80% validation attack success.
# If none meets 80%, select the highest observed validation success, and report the miss.
eligible = [r for r in validation_rows if r["validation_attack_success_rate"] >= TARGET_ATTACK_SUCCESS]
selected = (min(eligible, key=lambda r: r["epsilon"]) if eligible else
            max(validation_rows, key=lambda r: r["validation_attack_success_rate"]))
SELECTED_EPSILON = selected["epsilon"]

print(f"Selected FGSM epsilon (on validation): {SELECTED_EPSILON:.4f}")
print(f"Validation attack success: {selected['validation_attack_success_rate']:.2%}")
print("80% validation attack-success goal:", "MET" if eligible else "NOT MET")

## 10. FINAL attack results on the held-out test split

This test is run only **after** model and epsilon selection. The test statistics are genuine outcomes of the experiment; no target number is forced.

In [ ]:
rows = []
test_preds_by_epsilon = {}

for eps in EPSILONS:
    adv_probs = predict_with_fgsm(model_resnet, x_test, y_test, eps)
    pred = adv_probs.argmax(axis=1)
    test_preds_by_epsilon[eps] = pred
    rows.append({
        "epsilon": float(eps),
        "clean_accuracy": float(clean_test_acc),
        "attacked_accuracy": float(accuracy_score(y_test, pred)),
        "attack_success_rate": float(np.mean(
            pred[correct_clean_mask] != y_test[correct_clean_mask]
        )),
        "test_images": int(len(y_test)),
        "originally_correct_images": correct_clean_count,
    })

results_df = pd.DataFrame(rows)
selected_row = results_df.iloc[np.argmin(np.abs(results_df['epsilon'] - SELECTED_EPSILON))]

print("\n=================== FINAL TEST RESULTS ===================")
print(f"Clean classifier accuracy:         {clean_test_acc:.2%}")
print(f"FGSM epsilon selected:            {SELECTED_EPSILON:.4f}")
print(f"Accuracy AFTER FGSM:             {selected_row['attacked_accuracy']:.2%}")
print(f"FGSM attack success rate (ASR):  {selected_row['attack_success_rate']:.2%}")
print(f"Clean accuracy >= 80%:           {clean_test_acc >= 0.80}")
print(f"FGSM attack success >= 80%:      {selected_row['attack_success_rate'] >= 0.80}")
print("==========================================================")
print("\nFull test results:")
print(results_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## 11. Export results: CSV, JSON report, charts and confusion matrices

In [ ]:
from pathlib import Path
import json

EXPORT = Path("traffic_sign_results")
EXPORT.mkdir(exist_ok=True)

results_df.to_csv(EXPORT / "fgsm_test_results.csv", index=False)
validation_df.to_csv(EXPORT / "fgsm_validation_results.csv", index=False)

class_report = classification_report(
    y_test, clean_pred, target_names=CLASS_NAMES,
    zero_division=0, output_dict=True
)
pd.DataFrame(class_report).T.to_csv(EXPORT / "clean_classification_report.csv")

report = {
    "experiment": "Offline traffic-sign Residual CNN / FGSM",
    "classes": CLASS_NAMES,
    "model_selection_seed": int(best_seed),
    "validation_clean_accuracy": float(best_val_accuracy),
    "test_clean_accuracy": float(clean_test_acc),
    "selected_epsilon_validation_only": float(SELECTED_EPSILON),
    "test_accuracy_after_fgsm": float(selected_row['attacked_accuracy']),
    "test_attack_success_rate": float(selected_row['attack_success_rate']),
    "test_image_count": int(len(y_test)),
    "correct_clean_test_image_count": int(correct_clean_count),
    "clean_accuracy_80pct_goal_met": bool(clean_test_acc >= 0.8),
    "attack_success_80pct_goal_met": bool(selected_row['attack_success_rate'] >= 0.8),
    "notice": "Measured outcomes only; offline saved-image classifier attack, not live-camera attack.",
}
with open(EXPORT / "metrics.json", "w") as f:
    json.dump(report, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results_df.epsilon, results_df.attacked_accuracy * 100,
        marker="o", label="Accuracy after FGSM")
ax.plot(results_df.epsilon, results_df.attack_success_rate * 100,
        marker="s", label="FGSM attack success rate")
ax.axhline(80, linestyle="--", linewidth=1.2,
           color="gray", label="80% goal")
ax.axvline(SELECTED_EPSILON, linestyle=":", linewidth=1.0,
           color="gray", label="Selected epsilon")
ax.set(xlabel="Epsilon (pixels normalized to [0,1])",
       ylabel="Percentage (%)", ylim=(0, 105),
       title="Residual CNN — held-out traffic-sign test results")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
fig.savefig(EXPORT / "fgsm_results_chart.png", dpi=160)
plt.show()

selected_pred = test_preds_by_epsilon[SELECTED_EPSILON]
for name, predictions in [("clean", clean_pred), ("fgsm", selected_pred)]:
    cm = confusion_matrix(y_test, predictions, labels=np.arange(len(CLASS_NAMES)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=25)
    ax.set_title(f"Traffic sign confusion matrix — {name}")
    plt.tight_layout()
    fig.savefig(EXPORT / f"confusion_matrix_{name}.png", dpi=160)
    plt.show()

## 12. Show one camera-compatible example (offline)

The figure shows a saved/test image and its altered digital version. It does **not** perturb the webcam stream. It chooses an initially correct example that failed, when one exists, for illustration; the aggregate CSV reports every test example.

In [ ]:
changed = np.where(correct_clean_mask & (selected_pred != y_test))[0]
correct = np.where(correct_clean_mask)[0]
example_idx = int(changed[0] if len(changed) else correct[0])

sample_x = x_test[example_idx:example_idx+1]
sample_y = y_test[example_idx:example_idx+1]
adv_x = fgsm_attack_batch(model_resnet, sample_x, sample_y, SELECTED_EPSILON).numpy()
delta = adv_x[0] - sample_x[0]

if SELECTED_EPSILON > 0:
    visible_delta = np.clip(delta / (2 * SELECTED_EPSILON) + 0.5, 0, 1)
else:
    visible_delta = np.full_like(delta, 0.5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample_x[0])
axes[0].set_title(f"Original: {CLASS_NAMES[clean_pred[example_idx]]}")
axes[1].imshow(visible_delta)
axes[1].set_title(f"Amplified FGSM noise\nepsilon={SELECTED_EPSILON:.4f}")
axes[2].imshow(adv_x[0])
axes[2].set_title(f"After: {CLASS_NAMES[selected_pred[example_idx]]}")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
fig.savefig(EXPORT / "fgsm_example.png", dpi=180)
plt.show()

## 13. Save the trained model, class map, and download a results ZIP

The saved `.keras` classifier and JSON class names work with `residual_cnn_camera.py` from the previous package. No trained weights are included in the notebook; they are generated only after you run training.

In [ ]:
import zipfile

with open("residual_classes.json", "w") as f:
    json.dump(CLASS_NAMES, f)

results_zip = "traffic_sign_measured_results.zip"
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in EXPORT.rglob("*"):
        if p.is_file():
            z.write(p, p.as_posix())

print("Saved model:", BEST_MODEL_FILE)
print("Saved class map: residual_classes.json")
print("Results archive:", results_zip)

from google.colab import files
files.download(results_zip)
files.download(BEST_MODEL_FILE)
files.download("residual_classes.json")